In [1]:
import os
from google.colab import drive
drive.mount('/content/drive')
os.chdir("/content/drive/MyDrive/project_thinkfield/tinyrecursivemodels")
os.environ['DISABLE_COMPILE'] = '1'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install -r requirements.txt

In [26]:
import os
import json
from glob import glob
import hashlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import torch
import torch.nn.functional as F
import numpy as np
import yaml
from omegaconf import OmegaConf
from tqdm import tqdm
import math
import types
from typing import Dict
from numba import njit # grid_hash를 위해 유지

from models.recursive_reasoning.trm import TinyRecursiveReasoningModel_ACTV1Carry, TinyRecursiveReasoningModel_ACTV1InnerCarry
from puzzle_dataset import PuzzleDatasetMetadata
from pretrain import PretrainConfig, create_model, create_dataloader
from dataset.common import inverse_dihedral_transform

# --- 상수 정의 (동일) ---
DATASET_PATH = "data/arc_mini_arc2"
CHECKPOINT_PATH = "checkpoints/Arc_mini_arc2-ACT-torch/TinyRecursiveReasoningModel_ACTV1 thistle-giraffe/step_362160"
PAD_PUZZLE_IDENTIFIER = 0
ARC_COLOR_MAP = mcolors.ListedColormap([
    "#000000", "#0074D9", "#FF4136", "#2ECC40", "#FFDC00",
    "#AAAAAA", "#F012BE", "#FF851B", "#7FDBFF", "#870C25"
])

# --- 유틸리티 함수 (데이터 생성 로직과 일치하도록 최종 수정) ---

# 🚨 1. 색상 처리 오류를 해결한, 단순하고 올바른 inverse_aug
def inverse_aug(name: str, grid: np.ndarray):
    if "_" not in name:
        return grid
    try:
        trans_id, perm_str = name.split("_")[-2:]
        trans_id = int(trans_id[1:])
        perm = [int(p) for p in perm_str]
        inv_perm = np.argsort(perm)

        transformed_grid = inverse_dihedral_transform(grid, trans_id)
        # crop을 거친 0-9 범위의 그리드를 바로 처리
        return inv_perm[transformed_grid]
    except (ValueError, IndexError):
        return grid

def grid_hash(grid: np.ndarray):
    return hash((grid.tobytes(), grid.shape))

# 🚨 2. 데이터 생성 스크립트(build_arc_dataset.py)와 동일한 '경계 상자' crop 함수
def crop(grid: np.ndarray):
    if grid.ndim == 1:
        grid = grid.reshape(30, 30)

    # 색깔(2~11)이 있는 영역의 경계 상자를 찾음
    rows, cols = np.where((grid >= 2) & (grid <= 11))

    if rows.size == 0 or cols.size == 0:
        return np.empty((0, 0), dtype=np.uint8)

    cropped_grid = grid[min(rows):max(rows)+1, min(cols):max(cols)+1]

    # 시각화/평가를 위해 0~9 범위로 변환
    return (cropped_grid - 2).astype(np.uint8)

# --- 몽키 패치 코드 (동일) ---
def patched_initial_carry(self, batch: Dict[str, torch.Tensor]):
    batch_size = batch["inputs"].shape[0]
    device = next(self.parameters()).device
    inner_carry = TinyRecursiveReasoningModel_ACTV1InnerCarry(
        z_H=torch.empty(batch_size, self.inner.config.seq_len + self.inner.puzzle_emb_len, self.inner.config.hidden_size, dtype=self.inner.forward_dtype, device=device),
        z_L=torch.empty(batch_size, self.inner.config.seq_len + self.inner.puzzle_emb_len, self.inner.config.hidden_size, dtype=self.inner.forward_dtype, device=device)
    )
    return TinyRecursiveReasoningModel_ACTV1Carry(
        inner_carry=inner_carry,
        steps=torch.zeros((batch_size,), dtype=torch.int32, device=device),
        halted=torch.ones((batch_size,), dtype=torch.bool, device=device),
        current_data={k: torch.empty_like(v) for k, v in batch.items()}
    )

def patched_forward(self, carry: TinyRecursiveReasoningModel_ACTV1Carry, batch: dict):
    new_inner_carry = self.inner.reset_carry(carry.halted, carry.inner_carry)
    new_steps = torch.where(carry.halted, 0, carry.steps)
    new_current_data = {k: torch.where(carry.halted.view((-1,) + (1,) * (batch[k].ndim - 1)), batch[k], v) for k, v in carry.current_data.items()}
    new_inner_carry, logits, (q_halt_logits, q_continue_logits) = self.inner(new_inner_carry, new_current_data)
    outputs = {"logits": logits, "q_halt_logits": q_halt_logits, "q_continue_logits": q_continue_logits}
    with torch.no_grad():
        new_steps = new_steps + 1
        is_last_step = new_steps >= self.config.halt_max_steps
        halted = is_last_step
        if self.training and self.config.halt_max_steps > 1:
            if self.config.no_ACT_continue:
                halted = halted | (q_halt_logits > 0)
            else:
                halted = halted | (q_halt_logits > q_continue_logits)
    return TinyRecursiveReasoningModel_ACTV1Carry(
        inner_carry=new_inner_carry, steps=new_steps, halted=halted, current_data=new_current_data
    ), outputs

In [ ]:
Ks=[1, 2, 10, 100, 1000]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
with open(os.path.join(os.path.dirname(CHECKPOINT_PATH), "all_config.yaml"), "r") as f:
    config = PretrainConfig(**yaml.safe_load(f))
config.data_paths, config.data_paths_test = [DATASET_PATH], [DATASET_PATH]
save_outputs = ["inputs", "labels", "puzzle_identifiers", "logits", "q_halt_logits"]
print("Starting evaluation")
_, eval_metadata = create_dataloader(config, "test", test_set_mode=True, epochs_per_iter=1, global_batch_size=config.global_batch_size, rank=0, world_size=1)
eval_loader, _ = create_dataloader(config, "test", test_set_mode=True, epochs_per_iter=1, global_batch_size=config.global_batch_size, rank=0, world_size=1)
model, _, _ = create_model(config, eval_metadata, rank=0, world_size=1)
state_dict = torch.load(CHECKPOINT_PATH, map_location=device)
if any(key.startswith("_orig_mod.") for key in model.state_dict().keys()) and not any(key.startswith("_orig_mod.") for key in state_dict.keys()):
    state_dict = {"_orig_mod." + k: v for k, v in state_dict.items()}
model.load_state_dict(state_dict, strict=False)

model.model.initial_carry = types.MethodType(patched_initial_carry, model.model)
model.model.forward = types.MethodType(patched_forward, model.model)
model.to(device)
model.eval()

all_outputs = {key: [] for key in save_outputs}
eval_loader.dataset._lazy_load_dataset()
total_examples = sum(len(d["inputs"]) for d in eval_loader.dataset._data.values())
total_batches = math.ceil(total_examples / config.global_batch_size)
with torch.inference_mode():
    with tqdm(total=total_batches, desc="Evaluating") as pbar:
        for set_name, batch, global_batch_size in eval_loader:
            if global_batch_size == 0: continue
            pbar.set_description(f"Evaluating (Set: {set_name})")
            batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
            carry = model.initial_carry(batch)
            while True:
                carry, loss, metrics, preds, all_finish = model(carry=carry, batch=batch, return_keys=set(save_outputs))
                if all_finish: break
            outputs = {**batch, **preds}
            for key in save_outputs:
                if key in outputs:
                    all_outputs[key].append(outputs[key].cpu())
            pbar.update(1)


In [27]:
# Ks 변수가 혹시 초기화되었을 수 있으니 다시 정의
Ks=[1, 2, 10, 100, 1000]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# --- 후처리 (최종 수정된 함수와 올바른 파이프라인 적용) ---

# 1. 원시 예측 결과 취합 (기존과 동일)
all_preds = {key: torch.cat(all_outputs[key], dim=0) for key in save_outputs if all_outputs[key]}
with open(os.path.join(DATASET_PATH, "identifiers.json"), "r") as f:
    identifier_map = {i: v for i, v in enumerate(json.load(f))}
mask = all_preds["puzzle_identifiers"] != PAD_PUZZLE_IDENTIFIER
all_preds = {k: v[mask] for k, v in all_preds.items()}

# 2. 정답 데이터(puzzle_labels) 생성
global_hmap = {}
puzzle_labels = {}
for identifier, input_tensor, label_tensor in zip(all_preds["puzzle_identifiers"], all_preds["inputs"], all_preds["labels"]):
    name = identifier_map[identifier.item()]
    if "_" not in name:
        puzzle_labels.setdefault(name, {})
        input_np = crop(input_tensor.numpy())
        label_np = crop(label_tensor.numpy())
        input_hash = grid_hash(input_np)
        label_hash = grid_hash(label_np)
        global_hmap[input_hash] = input_np
        global_hmap[label_hash] = label_np
        if input_hash not in puzzle_labels[name]:
            puzzle_labels[name][input_hash] = label_hash
print(f"Number of original puzzles found: {len(puzzle_labels)}")

# 3. 예측 답변(pred_answers) 집계
preds = all_preds["logits"].argmax(-1)
pred_answers = {}
for identifier, input_tensor, pred_tensor, q_tensor in zip(all_preds["puzzle_identifiers"], all_preds["inputs"], preds, all_preds["q_halt_logits"].sigmoid()):
    name = identifier_map[identifier.item()]
    orig_name = name.split("_")[0]
    if orig_name not in puzzle_labels: continue

    # ✅ 올바른 순서: 1. crop 먼저 -> 2. inverse_aug 나중에
    # 입력(Input) 처리
    input_np_cropped = crop(input_tensor.numpy())
    input_np_processed = inverse_aug(name, input_np_cropped)
    input_hash = grid_hash(input_np_processed)
    if input_hash not in puzzle_labels[orig_name]: continue

    # 예측(Prediction) 처리
    pred_np_cropped = crop(pred_tensor.numpy())
    pred_np_processed = inverse_aug(name, pred_np_cropped)
    pred_hash = grid_hash(pred_np_processed)
    global_hmap[pred_hash] = pred_np_processed

    pred_answers.setdefault(orig_name, {})
    pred_answers[orig_name].setdefault(input_hash, [])
    pred_answers[orig_name][input_hash].append((pred_hash, q_tensor.item()))

print("Post-processing complete. Ready for final evaluation and visualization.")

Using device: cuda
Number of original puzzles found: 120
Post-processing complete. Ready for final evaluation and visualization.


In [28]:
# Ks 변수가 혹시 초기화되었을 수 있으니 다시 정의
Ks=[1, 2, 10, 100, 1000]

# --- 정확도 계산 및 시각화 (원본 로직 완벽 반영) ---
correct = [0] * len(Ks)

# tqdm을 사용하여 퍼즐 단위로 루프 실행
for puzzle_name, tests in tqdm(puzzle_labels.items(), desc="Evaluating and Visualizing Puzzles"):
    num_test_correct = [0] * len(Ks)

    # 시각화를 위한 figure 생성
    num_tests = len(tests)
    # 원본 시각화 로직은 Top-2 예측까지만 보여주므로 max_preds_to_show = 2
    max_preds_to_show = 2
    # squeeze=False 옵션으로 테스트 케이스가 1개일 때도 2차원 배열로 axes를 받도록 함
    fig, axes = plt.subplots(num_tests, 2 + max_preds_to_show, figsize=(4 * (2 + max_preds_to_show), num_tests * 4), squeeze=False)
    fig.suptitle(f"Puzzle: {puzzle_name}", fontsize=16)

    # 각 퍼즐에 포함된 테스트 케이스(input/label 쌍) 단위로 루프 실행
    for i, (input_hash, label_hash) in enumerate(tests.items()):
        p_map = {}
        # 해당 퍼즐, 해당 입력에 대한 예측값이 존재할 경우
        if puzzle_name in pred_answers and input_hash in pred_answers[puzzle_name]:
            p = pred_answers[puzzle_name][input_hash]

            # 1. 원본 방식대로 p_map 집계
            # p_map = { pred_hash: [득표수, 점수 총합], ... }
            for h, q in p:
                p_map.setdefault(h, [0, 0.0])
                p_map[h][0] += 1  # 득표수 (votes)
                p_map[h][1] += q  # 신뢰도 점수 (q_halt_logits) 총합

            # 2. 평균 점수 계산
            for h, stats in p_map.items():
                stats[1] /= stats[0] # 점수 총합 / 득표수

            # 3. 득표수 우선, 점수 차선으로 정렬
            p_map_sorted = sorted(p_map.items(), key=lambda kv: kv[1], reverse=True)

            # 4. Top-K 정확도 계산
            for j, k in enumerate(Ks):
                # 상위 k개 예측 중에 정답(label_hash)이 있는지 확인
                ok = any(h == label_hash for h, stats in p_map_sorted[:k])
                num_test_correct[j] += ok

            # --- 5. 시각화 ---
            # Input 이미지
            axes[i, 0].imshow(global_hmap[input_hash], cmap=ARC_COLOR_MAP, vmin=0, vmax=9)
            axes[i, 0].set_title("Input")
            axes[i, 0].axis('off')

            # 정답 (Correct Answer) 이미지
            axes[i, 1].imshow(global_hmap[label_hash], cmap=ARC_COLOR_MAP, vmin=0, vmax=9)
            axes[i, 1].set_title("Correct Answer")
            axes[i, 1].axis('off')

            # 상위 예측 (Top Predictions) 이미지
            for j, (h, stats) in enumerate(p_map_sorted[:max_preds_to_show]):
                pred_grid = global_hmap[h]
                axes[i, 2+j].imshow(pred_grid, cmap=ARC_COLOR_MAP, vmin=0, vmax=9)
                title = f"Pred #{j+1}\n(Votes: {stats[0]}, Score: {stats[1]:.2f})"
                # 예측이 정답과 같으면 녹색 테두리 표시
                if h == label_hash:
                    axes[i, 2+j].spines[:].set_color('green')
                    axes[i, 2+j].spines[:].set_linewidth(4)
                axes[i, 2+j].set_title(title)
                axes[i, 2+j].axis('off')

            # 예측 수가 max_preds_to_show 보다 적을 경우 남는 공간 비우기
            for j in range(len(p_map_sorted), max_preds_to_show):
                axes[i, 2+j].axis('off')

    # 퍼즐의 모든 테스트 케이스를 맞췄을 때만 최종 correct 카운트 증가
    for i in range(len(Ks)):
        if len(tests) > 0 and num_test_correct[i] == len(tests):
            correct[i] += 1

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

# 최종 정확도 출력
if len(puzzle_labels) > 0:
    print("\n--- Final Accuracy ---")
    for i, k in enumerate(Ks):
        print(f"Top-{k} Accuracy: {correct[i] / len(puzzle_labels) * 100:.2f}%")

Output hidden; open in https://colab.research.google.com to view.